# What is Adversarial Training?

Adversarial training is a defense strategy in which **adversarial examples are generated during training**, and the model is trained on these perturbed samples.  
By repeatedly exposing the model to attacks, it learns to become **robust** against them.

---

# Why Do We Need It?

Deep neural networks are highly vulnerable to **adversarial perturbations**—small, often imperceptible changes to the input that cause incorrect predictions.

Adversarial training helps the model:

- Learn a smoother and more stable loss landscape  
- Become more resistant to adversarial attacks  
- Improve robustness (usually at a slight cost to clean accuracy)

---

# PGD Adversarial Training (Madry et al.)

The most widely used and strongest baseline for adversarial robustness is **PGD adversarial training**. In this way, for each batch during training:

1. Start with a slightly perturbed version of the input  
2. Apply multiple PGD steps to craft a strong adversarial example  
3. Train the model using these adversarial samples  

This training update is commonly expressed as:
$
\theta_{t+1} = \theta_t - \eta \, \nabla_{\theta} \, L\big(f_{\theta}(x_{\text{adv}}),\, y\big)
$, where $x_{\text{adv}}$ was created by PGD.


# Adversarial Training Scenarios

Below are the different training strategies explored in this notebook. Each scenario demonstrates a distinct way of incorporating adversarial examples into the training process.

Let's begin by importing the necessary libraries and setting up initial configurations:




In [3]:
import copy
import math
import random
import time
from tqdm import tqdm
from typing import Tuple, Callable

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

In [4]:
# --- Reproducibility helpers ---
def set_seed(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Seed Setting (!!! do not change the seed !!!) ---
SEED = 2025
set_seed(SEED)

# Data / training hyperparameters
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

# 20 clean epochs, 20 std-adv epochs
EPOCHS_CLEAN = 20
EPOCHS_ADV_STD = 20      # standard PGD AT (for counting budget)
EPOCHS_ADV_MAX = 50      # max epochs for other ADV variants
EPOCHS_SEQ_CLEAN = 20    # clean phase epochs for sequential

EPS = 8 / 255.0
PGD_STEPS = 8
PGD_STEP_SIZE = 2 / 255.0
M = 4

In [5]:
# --- Data preparation ---
transform_train = transforms.Compose([
                                      transforms.RandomCrop(32, padding=4),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor()]
                                     )
transform_test = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

indices_train = torch.randperm(len(trainset), generator=torch.Generator().manual_seed(SEED))[:30000]
indices_test  = torch.randperm(len(testset), generator=torch.Generator().manual_seed(SEED))[:5000]

train_subset = Subset(trainset, indices_train)
test_subset = Subset(testset, indices_test)

trainloader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
testloader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:03<00:00, 49.3MB/s] 


In [7]:
# --- Model & Optimizer helpers ---
def make_model(num_classes=10):
    # TODO
    model = resnet18(pretrained=False)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

def make_optimizer(model):
    # TODO
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

    return optimizer

**Hint:** In our adversarial training setup, the goal of the budget system is to measure or constrain **only the expensive computations**, namely the gradient-based operations performed during PGD steps and the model’s backward pass. Since `.backward()` and `autograd.grad()` calls dominate the computational cost of adversarial training, the budget should be tied specifically to those operations.

In **standard adversarial training**, where we simply want to track how much compute is used without interrupting training, we use **budget_mode="count"** and call `budget_add` every time a gradient is computed.

In contrast, when we want to enforce a strict compute limit we use **budget_mode="consume"**, calling `budget_sub` after each gradient computation and stopping early once the budget is exhausted.



In [8]:
# --- Budget helpers ---
def make_budget(initial=0):
    """Create a mutable budget object."""
    return {"value": int(initial)}

def budget_add(budget, n=1):
    """Increases the budget counter."""
    if budget is not None:
        budget["value"] += int(n)

def budget_sub(budget, n=1):
    """Decreases the budget counter."""
    if budget is not None:
        budget["value"] -= int(n)

def budget_left(budget):
    """Returns the remaining budget, or None when budget is disabled."""
    return None if budget is None else int(budget["value"])

def budget_exhausted(budget):
    """Checks whether the budget is finished."""
    return (budget is not None) and (budget["value"] <= 0)

In [9]:
# --- PGD attack implementation ---
def pgd_attack(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = EPS,
    alpha: float = PGD_STEP_SIZE,
    steps: int = PGD_STEPS,
    budget=None,
    mode: str = None,
) -> torch.Tensor:
    """
    model : model to attack
    x : clean input batch
    y : labels for x
    eps : max L∞ perturbation
    alpha : PGD step size
    steps : number of PGD iterations
    budget : optional budget object for AT
    mode:
      - None      : ignore budget (for evaluation, etc.)
      - "count"   : budget++ for every autograd.grad (std adv training)
      - "consume" : budget-- for every autograd.grad, stop early if budget <= 0
    """
    x_adv = ...

    # TODO: Implement PGD attack with budget calculation
    x_adv = x.detach().clone()
    x_adv += torch.empty_like(x_adv).uniform_(-eps, eps)
    x_adv = torch.clamp(x_adv, 0, 1).detach()

    for step in range(steps):
        # Check budget if in consume mode
        if mode == "consume" and budget_exhausted(budget):
            break

        x_adv.requires_grad = True

        # Forward pass
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)

        # Compute gradient with respect to input
        grad = torch.autograd.grad(loss, x_adv, retain_graph=False, create_graph=False)[0]

        # Update budget based on mode
        if mode == "count":
            budget_add(budget, 1)
        elif mode == "consume":
            budget_sub(budget, 1)

        # PGD step: gradient ascent on input
        x_adv = x_adv.detach() + alpha * grad.sign()

        # Project back to eps ball around original input
        delta = torch.clamp(x_adv - x, -eps, eps)
        x_adv = torch.clamp(x + delta, 0, 1).detach()

    return x_adv

In [10]:
# --- Evaluation functions ---
def evaluate_clean(model: nn.Module, dataloader: DataLoader) -> Tuple[float, float]:
    model.eval()
    correct = 0
    total = 0

    # TODO: Evaluate the model on clean samples
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = correct / total
    return acc

def evaluate_adv(model: nn.Module, dataloader: DataLoader, attack_fn: Callable = pgd_attack, eps: float = EPS) -> Tuple[float, float]:
    model.eval()
    correct = 0
    total = 0

    # TODO: Apply PGD attack on clean data and evaluate the model on perturbed samples
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        # Generate adversarial examples (no budget tracking in evaluation)
        x_adv = attack_fn(model, x, y, eps=eps, mode=None)

        with torch.no_grad():
            logits = model(x_adv)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = correct / total
    return acc

In [11]:
# ---Prepare identical initialization ---
set_seed(SEED)
base_model = make_model().to(device)
initial_state = copy.deepcopy(base_model.state_dict())

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


## 1) Baseline: Train a Clean Model, Then Apply PGD Attack

In this scenario, we train a model **only on clean data**, with no adversarial examples involved.  
After training, we evaluate the model under a PGD attack.

In this scenario, our goal is to show how a standard (non-robust) model suffers *accuracy collapse* when exposed to strong adversarial perturbations. This serves as the reference point against which all adversarial training methods are compared.

In [12]:
def train_one_epoch_clean(model, loader, optimizer, budget=None):
    """
    Standard clean training on clean data only.
    """
    total_loss = 0.0

    # TODO: Implement one epoch clean training with budget calculation
    model.train()

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()

        # Count backward pass in budget if provided
        if budget is not None:
            budget_add(budget, 1)

        optimizer.step()
        total_loss += loss.item()

    total_loss = total_loss / len(loader)

    return total_loss

In [13]:
# Train clean model
model_clean = make_model().to(device)
model_clean.load_state_dict(copy.deepcopy(initial_state))
opt_clean = make_optimizer(model_clean)

print('Training clean baseline model...')
for ep in tqdm(range(1, EPOCHS_CLEAN + 1)):
    loss = train_one_epoch_clean(model_clean, trainloader, opt_clean)
    print(f"Epoch {ep}/{EPOCHS_CLEAN} | clean loss: {loss:.4f}")

torch.save(model_clean.state_dict(), "model_clean.pth")

clean_acc = evaluate_clean(model_clean, testloader)
print(f"\nClean model test accuracy: {clean_acc:.4f}")

Training clean baseline model...


  5%|▌         | 1/20 [00:17<05:37, 17.78s/it]

Epoch 1/20 | clean loss: 2.1135


 10%|█         | 2/20 [00:34<05:09, 17.20s/it]

Epoch 2/20 | clean loss: 1.5996


...

 95%|█████████▌| 19/20 [05:20<00:16, 16.81s/it]

Epoch 19/20 | clean loss: 0.4248


100%|██████████| 20/20 [05:37<00:00, 16.85s/it]

Epoch 20/20 | clean loss: 0.4200



Clean model test accuracy: 0.7388


In [14]:
adv_acc = evaluate_adv(model_clean, testloader, eps=8/255.0)
print(f"Under PGD attack (8/255) -> adversarial accuracy: {adv_acc:.4f}")

Under PGD attack (8/255) -> adversarial accuracy: 0.0000


### Now let's apply a weaker PGD attack to this cleanly trained model and observe the difference…

In [15]:
adv_acc_eps4 = evaluate_adv(model_clean, testloader, eps=4/255.0)
print(f"Under PGD attack (4/255) -> adversarial accuracy: {adv_acc_eps4:.4f}")

Under PGD attack (4/255) -> adversarial accuracy: 0.0012


In [16]:
adv_acc_eps2 = evaluate_adv(model_clean, testloader, eps=2/255.0)
print(f"Under PGD attack (2/255) -> adversarial accuracy: {adv_acc_eps2:.4f}")

Under PGD attack (2/255) -> adversarial accuracy: 0.0486


### **Question:** What can you infer from the adversarial accuracy presented above?
**Your Answer:** The clean model shows severe vulnerability to adversarial attacks, with accuracy dropping dramatically even under small perturbations, demonstrating the need for adversarial training.

Based on the results (Clean accuracy: 0.7388, PGD-8/255: 0.0000, PGD-4/255: 0.0012, PGD-2/255: 0.0486), we can infer:

1. **Complete vulnerability under strong attacks**: The model achieves 73.88% accuracy on clean data but drops to **0% accuracy** under the standard 8/255 PGD attack. This represents a **catastrophic failure** where the model cannot correctly classify even a single adversarial example. This demonstrates that deep neural networks, despite high clean performance, are extremely fragile to adversarial perturbations.

2. **Attack strength is critical**: Even when we reduce the perturbation budget:
   - At 4/255 (half the perturbation): adversarial accuracy is only 0.12% (still essentially zero)
   - At 2/255 (quarter perturbation): adversarial accuracy improves to 4.86%, but this is still a **93.4% drop** from clean accuracy
   
   This shows that even very small, imperceptible perturbations (2/255 ≈ 0.8% of the pixel range) can fool the model over 93% of the time.

3. **Fundamental lack of robustness**: Standard neural networks learn decision boundaries that are highly non-smooth and rely on brittle, non-robust features. They exploit spurious statistical correlations in the training data rather than learning semantically meaningful representations. This makes them trivially exploitable by gradient-based adversarial attacks.

4. **The robustness problem is severe**: The near-zero adversarial accuracy reveals that adversarial vulnerability is not just a minor issue but a fundamental problem that requires dedicated defense mechanisms like adversarial training to address.

This baseline clearly establishes the critical need for adversarial training techniques to build models that can maintain reasonable performance under adversarial attacks.



## 2) Standard Adversarial Training (PGD)

Using the **same initial weights as the clean model**, we train the model using **PGD-generated adversarial examples** at every step.

**How it works:**
1. For each batch, run PGD to generate $ x_{\text{adv}} $.  
2. Compute the loss on these adversarial samples.  
3. Backpropagate and update model parameters.

In this scenarion, we want to obtain the classical *PGD-adversarially trained model*.


In [17]:
def train_one_epoch_adv_standard(
    model,
    loader,
    optimizer,
    attack_fn=pgd_attack,
    budget=None,
    budget_mode=None,   # "count" or "consume" or None
):
    """
    Standard PGD adversarial training on adversarial examples only.
    model : neural network to train
    loader : training dataloader
    optimizer : optimizer used for updating model weights
    attack_fn : adversarial attack function (default: pgd_attack)
    budget : optional budget object for tracking compute
    budget_mode :
    - When budget_mode=="count": budget++ for autograd.grad in PGD and for backward().
    - When budget_mode=="consume": budget-- for both and stop when exhausted.
    """
    total_loss = 0.0

    # TODO: Implement Standard PGD adversarial training with budget calculation
    model.train()

    for x, y in loader:
        # Check if budget exhausted (for consume mode)
        if budget_mode == "consume" and budget_exhausted(budget):
            break

        x, y = x.to(device), y.to(device)

        # Generate adversarial examples
        x_adv = attack_fn(model, x, y, budget=budget, mode=budget_mode)

        # Train on adversarial examples
        optimizer.zero_grad()
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        loss.backward()

        # Count or consume budget for backward pass
        if budget_mode == "count":
            budget_add(budget, 1)
        elif budget_mode == "consume":
            budget_sub(budget, 1)

        optimizer.step()
        total_loss += loss.item()

    total_loss = total_loss / len(loader)

    return total_loss

In [18]:
# --- Standard adversarial training starting from SAME initial weights ---
model_adv_standard = make_model().to(device)
model_adv_standard.load_state_dict(copy.deepcopy(initial_state))
opt_adv_std = make_optimizer(model_adv_standard)
std_budget = make_budget(0) # This budget will COUNT all backward() and autograd.grad() calls

print('Training standard adversarial training (PGD) from same init...')
for ep in tqdm(range(1, EPOCHS_ADV_STD + 1)):
    loss = train_one_epoch_adv_standard(model_adv_standard, trainloader, opt_adv_std,
                                        attack_fn=pgd_attack, budget=std_budget, budget_mode="count")
    print(f"Epoch {ep}/{EPOCHS_ADV_STD} | adv loss: {loss:.4f}")

total_budget = budget_left(std_budget)
print(f"\nTotal gradient budget from standard PGD AT: {total_budget}")

torch.save(model_adv_standard.state_dict(), "model_adv_standard.pth")

Training standard adversarial training (PGD) from same init...


  5%|▌         | 1/20 [01:51<35:10, 111.07s/it]

Epoch 1/20 | adv loss: 2.3532


 10%|█         | 2/20 [03:42<33:18, 111.05s/it]

Epoch 2/20 | adv loss: 2.0877


...

Epoch 18/20 | adv loss: 1.5786


 95%|█████████▌| 19/20 [35:10<01:51, 111.03s/it]

Epoch 19/20 | adv loss: 1.5653


100%|██████████| 20/20 [37:01<00:00, 111.06s/it]

Epoch 20/20 | adv loss: 1.5514

Total gradient budget from standard PGD AT: 42300


In [20]:
acc_clean_std = evaluate_clean(model_adv_standard, testloader)
acc_adv_std = evaluate_adv(model_adv_standard, testloader, eps=8/255.0)
print(f"\nStandard adv-trained model -> clean acc: {acc_clean_std:.4f} | adv acc: {acc_adv_std:.4f}")


Standard adv-trained model -> clean acc: 0.6512 | adv acc: 0.3886


### Let's examine how the attack’s epsilon ball influences the robust model’s accuracy...


In [21]:
acc_adv_std_eps4 = evaluate_adv(model_adv_standard, testloader, eps=4/255.0)
print(f"Under PGD attack (4/255) -> adversarial accuracy: {acc_adv_std_eps4:.4f}")

Under PGD attack (4/255) -> adversarial accuracy: 0.5232


In [22]:
acc_adv_std_eps2 = evaluate_adv(model_adv_standard, testloader, eps=2/255.0)
print(f"Under PGD attack (2/255) -> adversarial accuracy: {acc_adv_std_eps2:.4f}")

Under PGD attack (2/255) -> adversarial accuracy: 0.5888


## 3) Joint Clean + Adversarial Loss

For each clean batch:
1. Generate adversarial examples using PGD or FGSM.  
2. Compute **clean loss** and **adversarial loss**.  
3. Backpropagate on their **mean**:
$
L_{\text{total}} = \frac{1}{2}\big(L(x, y) + L(x_{\text{adv}}, y)\big)
$

In this case we want to train a model that balances *clean accuracy* and *robust accuracy*, by learning from both types of data simultaneously.


In [23]:
def train_one_epoch_mean_clean_adv(
    model,
    loader,
    optimizer,
    attack_fn=pgd_attack,
    budget=None,
):
    """
    One backward per batch on 0.5*(loss_clean + loss_adv).
    In adv training (mean), we *consume* budget: both PGD grads and backward() are charged.
    """
    total_loss = 0.0

    # TODO: Implement PGD adversarial training based on the Mean Loss with budget calculation
    model.train()

    for x, y in loader:
        # Check if budget exhausted
        if budget_exhausted(budget):
            break

        x, y = x.to(device), y.to(device)

        # Generate adversarial examples (consumes budget)
        x_adv = attack_fn(model, x, y, budget=budget, mode="consume")

        if budget_exhausted(budget):
            break

        # Compute clean loss
        optimizer.zero_grad()
        logits_clean = model(x)
        loss_clean = F.cross_entropy(logits_clean, y)

        # Compute adversarial loss
        logits_adv = model(x_adv)
        loss_adv = F.cross_entropy(logits_adv, y)

        # Mean loss
        loss = 0.5 * (loss_clean + loss_adv)
        loss.backward()

        # Consume budget for backward
        budget_sub(budget, 1)

        optimizer.step()
        total_loss += loss.item()

    total_loss = total_loss / len(loader)

    return total_loss

In [24]:
# --- Mean (clean, adv) under fixed budget ---
model_mean = make_model().to(device)
model_mean.load_state_dict(copy.deepcopy(initial_state))
opt_mean = make_optimizer(model_mean)
mean_budget = make_budget(total_budget)

print('Training on mean(clean loss, adv loss) under fixed budget...')
for ep in tqdm(range(1, EPOCHS_ADV_MAX + 1)):
    if budget_exhausted(mean_budget):
        print(f"Budget exhausted before epoch {ep}.")
        break

    loss = train_one_epoch_mean_clean_adv(model_mean, trainloader, opt_mean,
                                          attack_fn=pgd_attack, budget=mean_budget)
    print(
        f"Epoch {ep}/{EPOCHS_ADV_MAX} | mean loss: {loss:.4f} "
        f"| budget left: {budget_left(mean_budget)}"
    )

    if budget_exhausted(mean_budget):
        print("Budget exhausted, stopping mean training.")
        break

torch.save(model_mean.state_dict(), "model_mean.pth")

Training on mean(clean loss, adv loss) under fixed budget...


  2%|▏         | 1/50 [02:06<1:43:40, 126.94s/it]

Epoch 1/50 | mean loss: 2.3115 | budget left: 40185


  4%|▍         | 2/50 [04:13<1:41:32, 126.93s/it]

Epoch 2/50 | mean loss: 1.9544 | budget left: 38070


...

Epoch 18/50 | mean loss: 1.2252 | budget left: 4230


 38%|███▊      | 19/50 [40:12<1:05:36, 126.99s/it]

Epoch 19/50 | mean loss: 1.2115 | budget left: 2115


 38%|███▊      | 19/50 [42:19<1:09:03, 133.65s/it]

Epoch 20/50 | mean loss: 1.2030 | budget left: 0
Budget exhausted, stopping mean training.


In [25]:
acc_clean_mean = evaluate_clean(model_mean, testloader)
acc_adv_mean = evaluate_adv(model_mean, testloader)
print(f"\nMean-loss model -> clean acc: {acc_clean_mean:.4f} | adv acc: {acc_adv_mean:.4f}")


Mean-loss model -> clean acc: 0.7188 | adv acc: 0.3476


## 4) Sequential Training: Clean First → Adversarial Later

Training is split into two distinct phases:

- **Phase 1:** Train the model normally on clean data only.  
- **Phase 2:** Continue training using only adversarial examples.

Here we start with a well-performing clean model, then adapt it to robustness.  


In [26]:
# --- Sequential training: clean phase then adversarial phase ---
model_seq = make_model().to(device)
model_seq.load_state_dict(copy.deepcopy(initial_state))
opt_seq = make_optimizer(model_seq)
seq_budget = make_budget(total_budget)

print('Sequential training (clean -> adv) under fixed budget...')

# 1) Clean phase: fixed number of epochs, consume budget per backward
for ep_clean in tqdm(range(1, EPOCHS_SEQ_CLEAN + 1)):
    if budget_exhausted(seq_budget):
        print(f"Budget exhausted during clean phase before epoch {ep_clean}.")
        break

    # loss_clean = train_one_epoch_clean(model_seq, trainloader, opt_seq) # we did not use this because it doesn't control budget!
    model_seq.train()
    loss_clean = 0.0

    for x, y in trainloader:
        if budget_exhausted(seq_budget):
            break

        x, y = x.to(device), y.to(device)

        opt_seq.zero_grad()
        logits = model_seq(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()

        # Consume budget for backward
        budget_sub(seq_budget, 1)

        opt_seq.step()
        loss_clean += loss.item()

    loss_clean /= len(trainloader)

    print(
        f"Sequential clean phase | epoch {ep_clean}/{EPOCHS_SEQ_CLEAN} "
        f"| loss: {loss_clean:.4f} | budget left: {budget_left(seq_budget)}"
    )

    if budget_exhausted(seq_budget):
        print("Budget exhausted at end of clean phase.")
        break

print('Switching to adversarial phase...')

# 2) Adversarial phase: PGD training consuming whatever budget is left
for ep_adv in tqdm(range(1, EPOCHS_ADV_MAX + 1)):
    if budget_exhausted(seq_budget):
        print(f"Budget exhausted before adversarial epoch {ep_adv}.")
        break

    loss_adv = train_one_epoch_adv_standard(model_seq, trainloader, opt_seq,
                                            attack_fn=pgd_attack, budget=seq_budget,
                                            budget_mode="consume")

    print(
        f"Sequential adv phase | epoch {ep_adv}/{EPOCHS_ADV_MAX} "
        f"| loss: {loss_adv:.4f} | budget left: {budget_left(seq_budget)}"
    )

    if budget_exhausted(seq_budget):
        print("Budget exhausted in adversarial phase, stopping.")
        break

torch.save(model_seq.state_dict(), "model_seq.pth")


Sequential training (clean -> adv) under fixed budget...


  5%|▌         | 1/20 [00:16<05:19, 16.79s/it]

Sequential clean phase | epoch 1/20 | loss: 1.9645 | budget left: 42065


 10%|█         | 2/20 [00:33<05:02, 16.78s/it]

Sequential clean phase | epoch 2/20 | loss: 1.5037 | budget left: 41830


...

Sequential adv phase | epoch 16/50 | loss: 1.4721 | budget left: 3760


 34%|███▍      | 17/50 [31:26<1:01:01, 110.94s/it]

Sequential adv phase | epoch 17/50 | loss: 1.4662 | budget left: 1645


 34%|███▍      | 17/50 [32:52<1:03:49, 116.05s/it]

Sequential adv phase | epoch 18/50 | loss: 1.1361 | budget left: -1
Budget exhausted in adversarial phase, stopping.


In [27]:
acc_clean_seq = evaluate_clean(model_seq, testloader)
acc_adv_seq = evaluate_adv(model_seq, testloader)
print(f"\nSequential model -> clean acc: {acc_clean_seq:.4f} | adv acc: {acc_adv_seq:.4f}")


Sequential model -> clean acc: 0.7062 | adv acc: 0.4026


## 5) Alternating Training: Clean Batch ↔ Adversarial Batch

During each epoch, training alternates between:

- one clean batch  
- one adversarial batch  
- one clean batch  
- one adversarial batch  
- …and so on

In this scenario, we expose the model to both clean and adversarial data *within the same training phase*, allowing it to maintain clean performance while becoming robust.

In [28]:
def train_one_epoch_alternating(model, loader, optimizer, budget=None):
    """
    For each batch:
      - one clean step
      - one adv step (with PGD)
    Both steps consume budget (PGD autograd.grad + both backwards).
    """
    total_loss_clean = 0.0
    total_loss_adv = 0.0

    # TODO: Implement PGD alternating training with budget calculation
    model.train()

    for x, y in loader:
        if budget_exhausted(budget):
            break

        x, y = x.to(device), y.to(device)

        # --- Clean step ---
        optimizer.zero_grad()
        logits_clean = model(x)
        loss_clean = F.cross_entropy(logits_clean, y)
        loss_clean.backward()

        # Consume budget for clean backward
        budget_sub(budget, 1)

        optimizer.step()
        total_loss_clean += loss_clean.item()

        if budget_exhausted(budget):
            break

        # --- Adversarial step ---
        x_adv = pgd_attack(model, x, y, budget=budget, mode="consume")

        if budget_exhausted(budget):
            break

        optimizer.zero_grad()
        logits_adv = model(x_adv)
        loss_adv = F.cross_entropy(logits_adv, y)
        loss_adv.backward()

        # Consume budget for adv backward
        budget_sub(budget, 1)

        optimizer.step()
        total_loss_adv += loss_adv.item()

    total_loss_clean = total_loss_clean / len(loader)
    total_loss_adv = total_loss_adv / len(loader)

    return (total_loss_clean, total_loss_adv)

In [29]:
# --- Alternating training: one clean batch, one adv batch ---
model_alt = make_model().to(device)
model_alt.load_state_dict(copy.deepcopy(initial_state))
opt_alt = make_optimizer(model_alt)
alt_budget = make_budget(total_budget)

print('Alternating training (clean + adv batches) under fixed budget...')
for ep in tqdm(range(1, EPOCHS_ADV_MAX + 1)):
    if budget_exhausted(alt_budget):
        print(f"Budget exhausted before epoch {ep}.")
        break

    clean_loss, adv_loss = train_one_epoch_alternating(model_alt, trainloader, opt_alt,
                                                        budget=alt_budget)

    print(
        f"Epoch {ep}/{EPOCHS_ADV_MAX} | clean loss: {clean_loss:.4f} "
        f"| adv loss: {adv_loss:.4f} | budget left: {budget_left(alt_budget)}"
    )

    if budget_exhausted(alt_budget):
        print("Budget exhausted, stopping alternating training.")
        break

torch.save(model_alt.state_dict(), "model_alt.pth")


Alternating training (clean + adv batches) under fixed budget...


  2%|▏         | 1/50 [02:07<1:43:46, 127.08s/it]

Epoch 1/50 | clean loss: 2.0538 | adv loss: 2.2782 | budget left: 39950


  4%|▍         | 2/50 [04:14<1:41:41, 127.12s/it]

Epoch 2/50 | clean loss: 1.6794 | adv loss: 2.0546 | budget left: 37600


...

Epoch 16/50 | clean loss: 0.9086 | adv loss: 1.6684 | budget left: 4700


 34%|███▍      | 17/50 [36:02<1:09:59, 127.26s/it]

Epoch 17/50 | clean loss: 0.8945 | adv loss: 1.6620 | budget left: 2350


 34%|███▍      | 17/50 [38:10<1:14:05, 134.71s/it]

Epoch 18/50 | clean loss: 0.8858 | adv loss: 1.6480 | budget left: 0
Budget exhausted, stopping alternating training.


In [30]:
acc_clean_alt = evaluate_clean(model_alt, testloader)
acc_adv_alt = evaluate_adv(model_alt, testloader)
print(f"\nAlternating model -> clean acc: {acc_clean_alt:.4f} | adv acc: {acc_adv_alt:.4f}")


Alternating model -> clean acc: 0.7358 | adv acc: 0.3346


In [31]:
# --- Summary (print final table) ---
from tabulate import tabulate

rows = [
    ['Baseline clean (no adv training)', f"{clean_acc:.4f}", f"{adv_acc:.4f}"],
    ['Standard PGD adv-train', f"{acc_clean_std:.4f}", f"{acc_adv_std:.4f}"],
    ['Mean(clean, adv)', f"{acc_clean_mean:.4f}", f"{acc_adv_mean:.4f}"],
    ['Sequential (clean -> adv)', f"{acc_clean_seq:.4f}", f"{acc_adv_seq:.4f}"],
    ['Alternating batches', f"{acc_clean_alt:.4f}", f"{acc_adv_alt:.4f}"],
]
print('\nFinal comparison table:')
print(tabulate(rows, headers=['Scenario', 'Clean Acc', 'Adv Acc']))


Final comparison table:
Scenario                            Clean Acc    Adv Acc
--------------------------------  -----------  ---------
Baseline clean (no adv training)       0.7388     0
Standard PGD adv-train                 0.6512     0.3886
Mean(clean, adv)                       0.7188     0.3476
Sequential (clean -> adv)              0.7062     0.4026
Alternating batches                    0.7358     0.3346


### **Question:** Based on the results from all above adversarial training scenarios, compare the key features of each method — including clean accuracy, adversarial accuracy (under PGD), robust–clean accuracy trade-off, and training stability. Which adversarial training strategy provides the best balance between clean and robust performance?

**Your Answer:** As we see in the above table the **'Sequential (clean -> adv)'** strategy provides the best balance between clean and robust performance with *'Clean Acc = 70.62%'* and  *'Adv Acc = 40.26%'*.


Based on the experimental results, here is a detailed comparison:

**1. Standard PGD Adversarial Training (Clean: 0.6512, Adv: 0.3886):**
- Achieves the **second-highest adversarial accuracy** (38.86%)
- Suffers from the **largest clean accuracy drop**: 73.88% → 65.12% (11.9% decrease)
- Most computationally expensive (serves as our budget baseline)
- Training is stable and straightforward with consistent convergence
- Strong robustness but at a considerable cost to clean performance
- Represents the traditional approach that prioritizes robustness over clean accuracy

**2. Mean (Clean + Adversarial) Loss (Clean: 0.7188, Adv: 0.3476):**
- **Second-highest clean accuracy** among adversarial training methods (71.88%)
- Achieves moderate adversarial accuracy (34.76%)
- Moderate clean accuracy retention: 2.7% drop from baseline (73.88% → 71.88%)
- Adversarial accuracy 10.6% lower than standard PGD adversarial training (34.76% vs 38.86%)
- Provides a reasonable balance through joint optimization of both objectives
- Training is stable since both objectives are optimized together
- Middle-ground performance on both metrics

**3. Sequential Training (Clean → Adversarial) (Clean: 0.7062, Adv: 0.4026):**
- **Achieves the highest adversarial accuracy** (40.26%) - the most robust model!
- Good clean accuracy (70.62%), with 4.4% drop from baseline
- Outperforms standard PGD adversarial training in robustness (40.26% vs 38.86%) while using the same budget
- The two-phase approach allows the model to first learn good representations, then adapt them for robustness
- **Best robustness-to-clean-accuracy ratio**: Achieves strongest robustness with only moderate clean accuracy sacrifice
- Training stability can vary at the phase transition point but overall very effective

**4. Alternating Training (Clean: 0.7358, Adv: 0.3346):**
- **Highest clean accuracy** among all adversarial training methods (73.58%), nearly matching baseline (only 0.4% drop)
- However, achieves the **lowest adversarial accuracy** (33.46%) among all adversarial training methods
- Excellent clean performance preservation but weakest robustness
- Clean accuracy drop of only 0.4% is impressive, but adversarial accuracy is 16.7% lower than sequential training
- The alternating approach preserves clean performance but fails to build strong robustness
- Training can be **unstable** due to switching between different objectives each batch
- **Poor efficiency**: Despite excellent clean accuracy, the computational budget is not effectively converted into robustness

**Key Insights:**

1. **Robustness-Accuracy Trade-off**: All adversarial training methods reduce clean accuracy compared to the baseline, but the magnitude varies significantly (0.4% to 11.9% drop). However, smaller clean accuracy drops don't guarantee better robustness—Alternating training has the smallest clean drop but weakest robustness.

2. **Sequential Training Dominates in Robustness**: Sequential training achieving the highest adversarial accuracy (40.26%) while maintaining strong clean accuracy (70.62%, only 4.4% drop) is remarkable. This 4.4% sacrifice yields the strongest robustness, making it the most **efficient** use of the clean-accuracy budget.

3. **Alternating Training's Paradox**: Despite achieving the highest clean accuracy (73.58%, only 0.4% drop from baseline), alternating training has the worst robustness (33.46%). This reveals a critical insight: preserving clean accuracy doesn't automatically translate to robustness. The frequent objective switching prevents the model from learning truly robust features, resulting in a model that performs well on clean data but remains vulnerable to attacks.

4. **Standard PGD adversarial training's Heavy Cost**: Traditional PGD adversarial training achieves good robustness (38.86%) but requires the largest clean accuracy sacrifice (11.9% drop). This makes it less efficient than sequential training, which achieves even better robustness (40.26%) with much smaller clean accuracy loss (4.4% drop).

5. **Efficiency Ranking** (Robustness gained per % of clean accuracy lost):
   - **Sequential**: 40.26% robustness for 4.4% clean loss = **9.15 ratio** (most efficient)
   - **Alternating**: 33.46% robustness for 0.4% clean loss = 83.65 ratio (but absolute robustness is lowest)
   - **Mean Loss**: 34.76% robustness for 2.7% clean loss = **12.87 ratio** (good efficiency)
   - **Standard PGD adversarial training**: 38.86% robustness for 11.9% clean loss = **3.27 ratio** (least efficient)

**Conclusion:**

The **best strategy depends on the application priorities**:

- **For maximum robustness** (recommended for most scenarios): **Sequential Training** (Clean→Adv) is the clear winner, achieving **40.26% adversarial accuracy** (highest) while maintaining **70.62% clean accuracy** (only 4.4% drop). This represents the **best robustness-to-clean-accuracy trade-off** and is ideal for security-critical applications where strong defense against adversarial attacks is essential.

- **For preserving clean accuracy above all**: **Alternating Training** achieves **73.58% clean accuracy** (nearly baseline performance, only 0.4% drop) but sacrifices robustness significantly (33.46%, lowest among adversarial methods). Only suitable when adversarial attacks are rare, mild, or when clean performance is absolutely critical and some robustness is acceptable but not prioritized.

- **For moderate balance with stable training**: **Mean (Clean + Adversarial) Loss** provides **71.88% clean accuracy** with **34.76% adversarial accuracy**. Good efficiency (12.87 ratio) with stable, straightforward training. Suitable for applications where both metrics matter roughly equally and training stability is important.

- **Standard PGD Training**: Achieves 65.12% clean accuracy and 38.86% adversarial accuracy—good robustness but the largest clean accuracy sacrifice (11.9% drop) and lowest efficiency (3.27 ratio). Less recommended given that sequential training achieves better robustness with smaller clean accuracy cost.

**Overall Recommendation**: **Sequential Training** provides the best overall performance, achieving the **strongest robustness (40.26%)** with only a **moderate clean accuracy sacrifice (4.4% drop to 70.62%)**. It successfully balances both objectives more efficiently than any other method, making it the **preferred choice for most practical adversarial training scenarios** where robustness is important. If clean accuracy must be preserved at nearly any cost, alternating training is an option, but be aware it provides significantly weaker defense against adversarial attacks.





# Adversarial Training For Free ([Shafahi et al.](https://arxiv.org/abs/1904.12843))

**Free Adversarial Training** is an efficient method designed to achieve adversarial robustness **without increasing training cost** compared to standard (clean) training.

Traditional adversarial training (especially PGD-based) is expensive because it requires multiple gradient steps **per batch** to generate adversarial examples.  
Free Adversarial Training avoids this overhead using two key ideas:

#### **1) Reuse the same batch multiple times**
Instead of performing many PGD steps *inside a single batch*, the same batch is processed for **multiple mini-steps** (called “free” steps), effectively simulating PGD iterations across repeated passes.

#### **2) Use gradient ascent on the input simultaneously with gradient descent on the weights**
Each training step does two things:

- **Update the adversarial example**  
  by performing gradient ascent on the input (using the backward pass you already computed)

- **Update the model parameters**  
  using the same backward pass (gradient descent as usual)

This means **one backward pass per step**, instead of many per PGD attack. Let's implement this method:


In [32]:
def train_one_epoch_free(
    model,
    loader,
    optimizer,
    m: int = 4,
    eps: float = EPS,
    alpha: float = PGD_STEP_SIZE,
    budget=None,
):
    total_loss = 0.0

    # TODO: Implement PGD adversarial training for free with budget calculation
    # Free adversarial training: reuse the same batch m times, 
    #   updating both the adversarial perturbation and model weights
    #   in each mini-step using a single backward pass.
    model.train()
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        # Initialize perturbation (randomly within eps ball)
        delta = torch.zeros_like(x).uniform_(-eps, eps)
        delta.requires_grad = True
        
        # Process same batch m times
        for _ in range(m):
            if budget_exhausted(budget):
                break
            
            # Create adversarial example
            x_adv = torch.clamp(x + delta, 0, 1)
            
            # Forward pass
            optimizer.zero_grad()
            logits = model(x_adv)
            loss = F.cross_entropy(logits, y)
            
            # Single backward pass - computes gradients for both model and delta
            loss.backward()
            
            # Consume budget for backward
            budget_sub(budget, 1)
            
            # Update model parameters (gradient descent)
            optimizer.step()
            
            # Update perturbation (gradient ascent on input)
            if delta.grad is not None:
                with torch.no_grad():
                    # Gradient ascent step
                    delta.data = delta.data + alpha * delta.grad.sign()
                    # Project back to eps ball
                    delta.data = torch.clamp(delta.data, -eps, eps)
                    # Zero out gradients for next iteration
                    delta.grad.zero_()
            
            total_loss += loss.item()
        
        if budget_exhausted(budget):
            break
    
    # Average over total mini-steps (batches * m)
    total_loss = total_loss / (len(loader) * m)

    return total_loss

In [33]:
# --- Free adversarial training under fixed budget ---
model_adv_free = make_model().to(device)
model_adv_free.load_state_dict(copy.deepcopy(initial_state))
opt_adv_free = make_optimizer(model_adv_free)
free_budget = make_budget(total_budget)

print(f'Training Free-AT with m={M} from same init under fixed budget...')
for ep in tqdm(range(1, EPOCHS_ADV_MAX + 1)):
    if budget_exhausted(free_budget):
        print(f"Budget exhausted before epoch {ep}.")
        break

    loss = train_one_epoch_free(model_adv_free, trainloader, opt_adv_free, 
                                m=M, eps=EPS, alpha=PGD_STEP_SIZE, budget=free_budget)
    print(
        f"Epoch {ep}/{EPOCHS_ADV_MAX} | free-adv loss: {loss:.4f} "
        f"| budget left: {budget_left(free_budget)}"
    )

    if budget_exhausted(free_budget):
        print("Budget exhausted, stopping Free-AT training.")
        break

torch.save(model_adv_free.state_dict(), "model_adv_free.pth")

Training Free-AT with m=4 from same init under fixed budget...


  2%|▏         | 1/50 [01:06<54:12, 66.39s/it]

Epoch 1/50 | free-adv loss: 2.0399 | budget left: 41360


  4%|▍         | 2/50 [02:12<53:05, 66.37s/it]

Epoch 2/50 | free-adv loss: 1.7326 | budget left: 40420


...

Epoch 43/50 | free-adv loss: 1.1112 | budget left: 1880


 88%|████████▊ | 44/50 [48:39<06:38, 66.39s/it]

Epoch 44/50 | free-adv loss: 1.1062 | budget left: 940


 88%|████████▊ | 44/50 [49:46<06:47, 67.87s/it]

Epoch 45/50 | free-adv loss: 1.1057 | budget left: 0
Budget exhausted, stopping Free-AT training.


In [34]:
acc_clean_free = evaluate_clean(model_adv_free, testloader)
acc_adv_free = evaluate_adv(model_adv_free, testloader)
print(f"\nAdversarial Training For Free model -> clean acc: {acc_clean_free:.4f} | adv acc: {acc_adv_free:.4f}")


Adversarial Training For Free model -> clean acc: 0.5532 | adv acc: 0.1296
